# Onwordly experiment run (Colab)

Runs any `.kaggle-run`-style plan on a Colab GPU. Default: Experiment 001 as designed (no format warm-up). Each `RESULTS.md` compares every regime against the untrained step-0 baseline, exact and lenient.

Runtime → Change runtime type → **T4 GPU**.

**Resumable:** results are written to Google Drive (`MyDrive/onwordly-results`). If the session drops, re-run all cells; finished regimes are loaded from Drive and only the unfinished one re-runs. Use a new `RESULTS_DIR` when you change a manifest.

In [ ]:
%cd /content
![ -d onwordly ] || git clone https://github.com/HereLiesAz/onwordly.git
%cd /content/onwordly
!git pull -q
!pip install -q -e '.[train]'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/onwordly-results'

In [ ]:
RUNS = {
    # name: (experiment, mode, manifest)
    '001': ('001', 'single', 'experiments/001-arithmetic-curriculum/manifest.json'),
}
SMOKE_FIRST = True  # run the 1k-token smoke manifest first to catch problems in minutes

In [ ]:
import subprocess

def run(name, experiment, mode, manifest):
    plan = f'/content/plan-{name}'
    open(plan, 'w').write(f'experiment: {experiment}\nmode: {mode}\nmanifest: {manifest}\n')
    subprocess.run(['onwordly-kaggle', '--plan', plan, '--output-root', f'{RESULTS_DIR}/{name}'], check=True)

if SMOKE_FIRST:
    run('smoke', '001', 'single', 'experiments/001-arithmetic-curriculum/smoke-manifest.json')
for name, (experiment, mode, manifest) in RUNS.items():
    run(name, experiment, mode, manifest)

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
for report in sorted(Path(RESULTS_DIR).glob('*/*/RESULTS.md')):
    display(Markdown(f'### {report.parent.parent.name}\n\n' + report.read_text()))

In [ ]:
# Results already live on Drive; this optional cell also downloads a zip of RESULTS_DIR.
import shutil
from google.colab import files
archive = shutil.make_archive('/content/results', 'zip', RESULTS_DIR)
files.download(archive)